In [4]:
import csv
import os
import pandas as pd

# Lista de arquivos CSV e nomes das instâncias
arquivos = [
    ("percurso_aula.csv", "Instância de Teste", "tab:percurso_aula"),
    ("percurso_gr15.tsp.csv", "Instância gr15.tsp", "tab:percurso_gr15"),
    ("percurso_gr17.tsp.csv", "Instância gr17.tsp", "tab:percurso_gr17"),
    ("percurso_gr21.tsp.csv", "Instância gr21.tsp", "tab:percurso_gr21"),
    ("percurso_gr24.tsp.csv", "Instância gr24.tsp", "tab:percurso_gr24"),
]

pasta = "../relatorio/resultado"
saida = os.path.join(pasta, "tabelas_latex.txt")

linhas_por_tabela = 5  # número de colunas por linha na tabela

for arquivo, nome, label in arquivos:
    caminho = os.path.join(pasta, arquivo)
    linhas = []
    with open(caminho, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            origem = row["Origem"]
            destino = row["Destino"]
            peso = row["Peso"]
            emp = row["Emparelhado"].strip().lower()
            celula = f"{origem} -- ({peso}) $\\rightarrow$ {destino}"
            if emp == "sim":
                celula = f"\\textcolor{{red}}{{{celula}}}"
            linhas.append(celula)
    # Quebra em linhas de 5 colunas, preenchendo vazios no final
    tabela = []
    for i in range(0, len(linhas), linhas_por_tabela):
        linha_celulas = linhas[i:i+linhas_por_tabela]
        if len(linha_celulas) < linhas_por_tabela:
            linha_celulas += [" "] * (linhas_por_tabela - len(linha_celulas))
        tabela.append("    " + " & ".join(linha_celulas) + " \\\\")
    # Monta LaTeX
    nome_tex = os.path.splitext(arquivo)[0] + ".tex"
    caminho_saida = os.path.join(pasta, nome_tex)
    with open(caminho_saida, "w", encoding="utf-8") as fsaida:
        fsaida.write("\\begin{table}[H]\n    \\centering\n    \\scriptsize\n    \\begin{tabular}{|c|c|c|c|c|}\n        \\hline\n        \\multicolumn{5}{|c|}{Percurso} \\\\ \\hline\n")
        for linha in tabela:
            fsaida.write(f"{linha}\n")
        fsaida.write("        \\hline\n    \\end{tabular}\n")
        fsaida.write(f"    \\caption{{Percurso ótimo encontrado para a {nome}. Células em vermelho indicam arestas emparelhadas pelo algoritmo.}}\n    \\label{{{label}}}\n\\end{{table}}\n")
    print(f"Tabela LaTeX gerada: {caminho_saida}")

# Gera tabela LaTeX do arquivo único de resultados
print("\n--- Gerando tabela LaTeX dos resultados ---")
caminho_resultados = os.path.join(pasta, "resultados.csv")

if os.path.exists(caminho_resultados):
    df = pd.read_csv(caminho_resultados)
    
    # Cria tabela LaTeX com as informações principais
    caminho_resultados_tex = os.path.join(pasta, "resumo_resultados.tex")
    with open(caminho_resultados_tex, "w", encoding="utf-8") as f:
        f.write("\\begin{table}[H]\n")
        f.write("    \\centering\n")
        f.write("    \\footnotesize\n")
        f.write("    \\begin{tabular}{|l|c|c|c|c|c|c|c|c|}\n")
        f.write("        \\hline\n")
        f.write("        \\textbf{Inst.} & \\textbf{Dim} & \\textbf{Arestas} & \\textbf{Custo Orig.} & \\textbf{Custo Final} & \\textbf{Vér. Ím.} & \\textbf{Arestas Dup.} & \\textbf{Overhead} & \\textbf{Overhead \\%} \\\\ \\hline\n")
        
        for idx, row in df.iterrows():
            instancia = row['Instância']
            dimensao = int(row['Dimensão (Vértices)'])
            arestas = int(row['Arestas Originais'])
            custo_orig = int(row['Custo Original'])
            custo_final = int(row['Custo Final'])
            verts_impar = int(row['Vértices Grau Ímpar'])
            arestas_dup = int(row['Arestas Duplicadas'])
            overhead = int(row['Overhead'])
            overhead_perc = row['Overhead (%)']
            
            f.write(f"        {instancia} & {dimensao} & {arestas} & {custo_orig} & {custo_final} & {verts_impar} & {arestas_dup} & {overhead} & {overhead_perc}\\% \\\\ \\hline\n")
        
        f.write("    \\end{tabular}\n")
        f.write("    \\caption{Resumo integrado dos resultados: dimensão do grafo, arestas originais, custos original e final, identificação de vértices de grau ímpar, arestas duplicadas pelo emparelhamento, e overhead computacional em valor absoluto e percentual.}\n")
        f.write("    \\label{tab:resumo_resultados}\n")
        f.write("\\end{table}\n")
    
    print(f"Tabela LaTeX gerada: {caminho_resultados_tex}")
    print("\nDados:")
    print(df[['Instância', 'Dimensão (Vértices)', 'Arestas Originais', 'Custo Original', 'Custo Final', 'Vértices Grau Ímpar', 'Arestas Duplicadas', 'Overhead', 'Overhead (%)']])
else:
    print(f"Arquivo não encontrado: {caminho_resultados}")


Tabela LaTeX gerada: ../relatorio/resultado\percurso_aula.tex
Tabela LaTeX gerada: ../relatorio/resultado\percurso_gr15.tsp.tex
Tabela LaTeX gerada: ../relatorio/resultado\percurso_gr17.tsp.tex
Tabela LaTeX gerada: ../relatorio/resultado\percurso_gr21.tsp.tex
Tabela LaTeX gerada: ../relatorio/resultado\percurso_gr24.tsp.tex

--- Gerando tabela LaTeX dos resultados ---
Tabela LaTeX gerada: ../relatorio/resultado\resumo_resultados.tex

Dados:
  Instância  Dimensão (Vértices)  Arestas Originais  Custo Original  \
0  aula.tsp                    7                 10              72   
1  gr15.tsp                   15                 40            1871   
2  gr17.tsp                   17                 81           22061   
3  gr21.tsp                   21                113           38429   
4  gr24.tsp                   24                116           16541   

   Custo Final  Vértices Grau Ímpar  Arestas Duplicadas  Overhead  \
0           95                    4                   2    